# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading and exploration of the FAIR² dataset package using the [mlcroissant](https://github.com/mlcommons/croissant) library.

## Dataset Source
The dataset source is a Croissant schema, available at:

**https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json**

In [ ]:
# Install latest mlcroissant if needed
!pip install --quiet mlcroissant

## 1. Data Loading
We'll load metadata and tables from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema. All entities are referenced by their `@id`.

In [ ]:
# Display all record sets with their `@id` and names
record_sets = dataset.record_sets
if record_sets:
    print("Record Sets:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")

    # Show available fields and columns for each record set
    for rs in record_sets:
        print(f"\nFields for record set {rs['@id']}:")
        if 'fields' in rs:
            for field in rs['fields']:
                print(f"  - Field @id: {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")

        if 'columns' in rs:
            print("  Columns:")
            for col in rs['columns']:
                print(f"    - Column @id: {col['@id']} | name: {col.get('name', 'N/A')} | dataType: {col.get('dataType', 'N/A')}")
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load all tabular data from available record sets using their `@id`s. All references use `@id` fields.

We'll inspect, then load, and show the columns for each record set.

In [ ]:
# Extract data from each record set
# Get record set `@id`s
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns (@id) in this record set:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Process, filter, and transform data. Operations include filtering numeric fields, normalization, and grouping. All columns referenced strictly via their `@id` (not name).

In [ ]:
# Choose a record set and fields for EDA
# Example: Use first available record set (edit as needed)

if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Try to find a numeric field by schema
    numeric_field_id = None
    for rs in dataset.record_sets:
        if rs['@id'] == record_set_id:
            for field in rs.get('fields', []):
                if field.get('dataType', '').lower() in ['integer', 'number', 'float']:
                    numeric_field_id = field['@id']
                    break
            break
    # If no numeric field found, fallback to column with integer or float dtype
    if not numeric_field_id:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    if numeric_field_id and numeric_field_id in df.columns:
        # Set threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping by a non-numeric field
        group_field_id = None
        for rs in dataset.record_sets:
            if rs['@id'] == record_set_id:
                for field in rs.get('fields', []):
                    if field.get('dataType', '') in ['Text', 'String', 'text', 'string']:
                        group_field_id = field['@id']
                        break
                break

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No record sets loaded.")

## 5. Visualization
Visualize selected numeric field distributions and relationships. Always reference fields and columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: histogram and boxplot for the chosen numeric field
if record_set_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    plt.figure(figsize=(7, 3))
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping field exists, visualize relationship
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(9, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we have:

- Loaded the FAIR² dataset package using `mlcroissant`
- Reviewed its metadata, record sets, and fields via their `@id`
- Extracted and displayed tabular data
- Performed basic filtering, normalization, and grouping by field IDs
- Visualized distributions and relationships by their Croissant `@id`

This workflow ensures transparent and reproducible access to biomedical research datasets using advanced schema standards.